In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL
from PyEMD import EMD  # pip install EMD-signal 필요

In [ ]:
def estimate_period_fft(series, default_period=24):
    """
    FFT를 활용하여 시계열 데이터에서 가장 지배적인 주기를 동적으로 추정합니다.
    """
    n = len(series)
    t = np.arange(n)
    
    # 1. FFT 전처리: 선형 추세 및 평균(DC 성분) 제거
    p = np.polyfit(t, series, 1)
    detrended = series - np.polyval(p, t)
    detrended = detrended - np.mean(detrended)
    
    # 2. FFT 수행 및 진폭 계산
    fft_vals = np.fft.rfft(detrended)
    frequencies = np.fft.rfftfreq(n)
    magnitudes = np.abs(fft_vals)
    
    # 3. 0Hz(상수항)를 제외하고 진폭이 가장 큰 인덱스 탐색
    if len(magnitudes) > 1:
        dominant_idx = np.argmax(magnitudes[1:]) + 1 
        dominant_freq = frequencies[dominant_idx]
        
        if dominant_freq > 0:
            period = int(np.round(1.0 / dominant_freq))
            if 2 <= period <= n // 2:
                return period
                
    return default_period

def calculate_3d_metrics(series):
    """
    FFT 기반 동적 주기 추출 후, STL 분해와 EMD를 결합하여 
    독립적 신호 대 잡음비(SNR) 기반의 F_T, F_S, F_I 지표를 산출합니다.
    """
    estimated_p = estimate_period_fft(series)
    
    if len(series) < 2 * estimated_p:
        estimated_p = max(2, len(series) // 2 - 1)
        
    try:
        # 1. STL 분해 (T, S, R_stl 도출)
        res = STL(series, period=estimated_p, robust=True).fit()
        T, S, R_stl = res.trend, res.seasonal, res.resid
        
        # 2. EMD 분해 (Residual을 다시 분해하여 Intervention 추출)
        emd = EMD()
        imfs = emd.emd(R_stl)
        
        # IMF 성분 중 노이즈(IMF1)와 최저주파 잔차를 제외한 중간 대역을 I(Intervention)로 정의
        if len(imfs) > 2:
            I = np.sum(imfs[1:-1], axis=0) 
        elif len(imfs) == 2:
            I = imfs[1]
        else:
            I = np.zeros_like(R_stl)
            
        # ★ 3. 순수 노이즈(R_pure) 도출: STL 잔차에서 EMD로 찾은 개입 성분(I) 차감
        R_pure = R_stl - I
        
        # ★ 4. 3차원 강도 산출 (독립적 신호 대 잡음비 방식 적용)
        var_R = np.var(R_pure)
        
        var_TR = np.var(T + R_pure)
        var_SR = np.var(S + R_pure)
        var_IR = np.var(I + R_pure)
        
        # Hyndman 공리 기반의 성분별 직교(Orthogonal) 강도 계산
        F_T = max(0.0, 1.0 - (var_R / var_TR if var_TR > 0 else 1.0))
        F_S = max(0.0, 1.0 - (var_R / var_SR if var_SR > 0 else 1.0))
        F_I = max(0.0, 1.0 - (var_R / var_IR if var_IR > 0 else 1.0))
        
    except Exception:
        # 분해 실패 시 예외 처리 (정상성 노이즈로 간주)
        F_T, F_S, F_I = 0.0, 0.0, 0.0
        
    return F_T, F_S, F_I, estimated_p

def generate_extensive_synthetic_dataset(n_samples_per_q=50, length=500):
    """
    4사분면 검증을 위해 통계적 특성이 다양한 복잡한 가상 시계열 데이터를 대량 생성합니다.
    """
    data_records = []
    t = np.arange(length)
    
    for i in range(n_samples_per_q):
        print(f"💡 Generating sample {i+1}/{n_samples_per_q} for each regime...")
        
        rand_p1 = np.random.choice([12, 24, 36, 48])
        rand_p2 = rand_p1 * 2
        
        # --- R1: Composite Regime ---
        slope = np.random.uniform(0.1, 0.4) * np.random.choice([1, -1])
        curve = np.random.uniform(1e-5, 5e-5) * np.random.choice([1, -1])
        amp1, amp2 = np.random.uniform(8, 25), np.random.uniform(2, 8)
        noise_std = np.random.uniform(0.5, 1.5)
        
        q1_series = (slope * t + curve * (t**2)) + (amp1 * np.sin(2 * np.pi * t / rand_p1)) + (amp2 * np.cos(2 * np.pi * t / rand_p2)) + np.random.normal(0, noise_std, length)
        q1_scaled = (q1_series - np.min(q1_series)) / (np.max(q1_series) - np.min(q1_series) + 1e-9)
        ft, fs, fi, p_est = calculate_3d_metrics(q1_scaled)
        
        data_records.append({'True_Regime': 'R1', 'Pattern': 'Trend+MultiSeason', 'F_T': ft, 'F_S': fs, 'F_I': fi, 'True_P': rand_p1, 'Est_P': p_est})
        
        # --- R2: Pure Seasonal ---
        amp1, amp2 = np.random.uniform(10, 30), np.random.uniform(4, 12)
        noise_std = np.random.uniform(0.5, 2.0)
        
        q2_series = (amp1 * np.sin(2 * np.pi * t / rand_p1)) + (amp2 * np.cos(2 * np.pi * t / rand_p2)) + np.random.normal(0, noise_std, length)
        q2_scaled = (q2_series - np.min(q2_series)) / (np.max(q2_series) - np.min(q2_series) + 1e-9)
        ft, fs, fi, p_est = calculate_3d_metrics(q2_scaled)
        
        data_records.append({'True_Regime': 'R2', 'Pattern': 'Pure_MultiSeason', 'F_T': ft, 'F_S': fs, 'F_I': fi, 'True_P': rand_p1, 'Est_P': p_est})
        
        # --- R3: Stationary / Noise ---
        noise_type = np.random.choice(['WN', 'AR'])
        if noise_type == 'WN':
            q3_series = np.random.normal(0, np.random.uniform(5, 15), length)
        else:
            phi_val = np.random.uniform(0.4, 0.7)
            q3_series = np.zeros(length)
            innovations = np.random.normal(0, 5.0, length)
            q3_series[0] = innovations[0]
            for idx in range(1, length):
                q3_series[idx] = phi_val * q3_series[idx-1] + innovations[idx]
                
        q3_scaled = (q3_series - np.min(q3_series)) / (np.max(q3_series) - np.min(q3_series) + 1e-9)
        ft, fs, fi, p_est = calculate_3d_metrics(q3_scaled)
        
        data_records.append({'True_Regime': 'R3', 'Pattern': f'Stationary_{noise_type}', 'F_T': ft, 'F_S': fs, 'F_I': fi, 'True_P': 'None', 'Est_P': p_est})

        # --- R4: Pure Trending (with Structural Breaks & Drifts) ---
        trend_type = np.random.choice(['Break', 'RandomWalk', 'Exponential'])
        noise_std = np.random.uniform(0.5, 1.5)
        
        if trend_type == 'Break':
            bp_val = length // 2
            q4_series = np.random.normal(0, noise_std, length)
            q4_series[bp_val:] += np.random.uniform(50, 150)
        elif trend_type == 'RandomWalk':
            drift_val = np.random.uniform(0.1, 0.5) * np.random.choice([1, -1])
            q4_series = np.cumsum(np.random.normal(drift_val, 2, length))
        else:
            q4_series = np.exp(np.linspace(0, np.random.uniform(3, 5), length)) + np.random.normal(0, noise_std, length)

        q4_scaled = (q4_series - np.min(q4_series)) / (np.max(q4_series) - np.min(q4_series) + 1e-9)
        ft, fs, fi, p_est = calculate_3d_metrics(q4_scaled)
        
        data_records.append({'True_Regime': 'R4', 'Pattern': f'PureTrend_{trend_type}', 'F_T': ft, 'F_S': fs, 'F_I': fi, 'True_P': 'None', 'Est_P': p_est})
        
    return pd.DataFrame(data_records)


In [ ]:
# ==========================================
# 1. 데이터 생성 실행 (테스트용 샘플 10개)
# ==========================================
df_exp_results = generate_extensive_synthetic_dataset(n_samples_per_q=3000, length=500)

In [ ]:
# ==========================================
# 2. 결과 시각화 (3D Cube Mapping)
# ==========================================
plt.rcParams['font.family'] = 'Malgun Gothic' # 환경에 따라 'AppleGothic' 등으로 변경 필요
plt.rcParams['axes.unicode_minus'] = False

fig = plt.figure(figsize=(14, 11))
ax = fig.add_subplot(111, projection='3d')

# 패턴별 색상 및 마커
pattern_configs = {
    'Trend+MultiSeason':       {'color': '#d63031', 'marker': 'o', 'label': 'R1: Composite (Trend+Season)'},
    'Pure_MultiSeason':        {'color': '#0984e3', 'marker': 'o', 'label': 'R2: Seasonal (Pure Season)'},
    'Stationary_WN':           {'color': '#2ed573', 'marker': 's', 'label': 'R3: Stationary (White Noise)'},
    'Stationary_AR':           {'color': '#009432', 'marker': '^', 'label': 'R3: Stationary (AR)'},
    'PureTrend_Break':         {'color': '#e67e22', 'marker': 'D', 'label': 'R4: Trending (Struct Break)'},
    'PureTrend_RandomWalk':    {'color': '#f1c40f', 'marker': 'v', 'label': 'R4: Trending (Random Walk)'},
    'PureTrend_Exponential':   {'color': '#8e44ad', 'marker': '*', 'label': 'R4: Trending (Exponential)'}
}

legend_order = list(pattern_configs.keys())
existing_patterns = [p for p in legend_order if p in df_exp_results['Pattern'].unique()]

# 3D 산점도 플로팅
for pattern_label in existing_patterns:
    group = df_exp_results[df_exp_results['Pattern'] == pattern_label]
    config = pattern_configs[pattern_label]
    
    ax.scatter(group['F_T'], group['F_S'], group['F_I'], 
               label=config['label'], color=config['color'], marker=config['marker'],
               alpha=0.8, edgecolors='k', s=85)

# 축 레이블 및 세팅
ax.set_xlabel('Trend Strength ($F_T$)', fontsize=12, fontweight='bold', labelpad=10)
ax.set_ylabel('Seasonal Strength ($F_S$)', fontsize=12, fontweight='bold', labelpad=10)
ax.set_zlabel('Intervention Strength ($F_I$)', fontsize=12, fontweight='bold', labelpad=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_zlim(0, 1)
ax.set_title('3D Regime Mapping Cube ($F_T$, $F_S$, $F_I$)', fontsize=16, fontweight='bold', pad=20)

# 범례 설정 (그래프 바깥쪽)
ax.legend(loc='center left', bbox_to_anchor=(1.1, 0.5), fontsize=10, framealpha=0.95, facecolor='white', edgecolor='gray')

plt.tight_layout()
plt.show()

# ==========================================
# 3. 간단한 리포트 (F_I 지표 평균 확인용)
# ==========================================
print("\n" + "="*60)
print("     [오라클 검증 통계 리포트 (3D 확장판 - 직교성 반영)]")
print("="*60)
for r_label in ['R1', 'R2', 'R3', 'R4']:
    sub_df = df_exp_results[df_exp_results['True_Regime'] == r_label]
    print(f"▶ {r_label} Regime 평균 지표값")
    print(f"   F_T: {sub_df['F_T'].mean():.3f} | F_S: {sub_df['F_S'].mean():.3f} | F_I: {sub_df['F_I'].mean():.3f}")
    if r_label == 'R4':
        print("   -> (R4 내부 상세 F_I 분석)")
        for p in sub_df['Pattern'].unique():
            p_df = sub_df[sub_df['Pattern'] == p]
            print(f"      {p}: F_I = {p_df['F_I'].mean():.3f}")
print("="*60)